# 🫀 CardioSense AI — Pipeline Testing Notebook

This notebook tests every component of the ML pipeline on real data.
Run cells top to bottom. Each section is independent and clearly labelled.

**What we test here:**

1. Data loading and validation
2. Preprocessing + feature engineering
3. All 6 classical ML models
4. PyTorch CardioNet DNN
5. SHAP explainability
6. Single patient prediction walkthrough


## 1 — Setup & Install


In [ ]:
# Install everything. Takes ~2 minutes on a fresh Colab runtime.
!pip install scikit-learn imbalanced-learn xgboost lightgbm shap torch seaborn -q
print('All packages installed.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics         import (accuracy_score, roc_auc_score, f1_score,
                                     roc_curve, confusion_matrix, classification_report)
from sklearn.calibration     import CalibratedClassifierCV
from imblearn.over_sampling  import SMOTE
import xgboost  as xgb
import lightgbm as lgb
import shap

print('Imports done.')

## 2 — Load Real Dataset


In [ ]:
# Option A: Download directly from UCI via sklearn
from sklearn.datasets import fetch_openml
heart = fetch_openml(name='heart-disease-ohio', version=1, as_frame=True, parser='auto')
df    = heart.frame
# Rename target column to 'target' and binarise (>0 = disease)
df['target'] = (df['num'].astype(int) > 0).astype(int)
df = df.drop(columns=['num'])
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Option B: Upload Kaggle CSV manually
# Uncomment if Option A fails, then upload your heart.csv via the Files panel

# from google.colab import files
# uploaded = files.upload()   # triggers file picker
# df = pd.read_csv('heart.csv')
# df.columns = [c.strip().lower() for c in df.columns]
# print(f'Uploaded dataset: {df.shape}')

In [ ]:
print('Shape:', df.shape)
print('\nClass distribution:')
print(df['target'].value_counts())
print('\nNull counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nData types:')
print(df.dtypes)

## 3 — Exploratory Data Analysis


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()

num_cols = df.select_dtypes(include=np.number).columns.tolist()

for i, col in enumerate(num_cols[:14]):
    for cls, color, lbl in [(0,'#38A169','No Disease'), (1,'#E53E3E','Disease')]:
        subset = df[df['target'] == cls][col].dropna()
        axes[i].hist(subset, bins=20, alpha=0.55, color=color, label=lbl)
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)

for j in range(len(num_cols), len(axes)):
    axes[j].axis('off')

plt.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# correlation heatmap
numeric_df = df.select_dtypes(include=np.number)
corr       = numeric_df.corr()

plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4 — Preprocessing & Feature Engineering


In [ ]:
BASE_FEATURES = [
    'age','sex','cp','trestbps','chol',
    'fbs','restecg','thalach','exang',
    'oldpeak','slope','ca','thal'
]

def fill_missing(df):
    df = df.copy()
    for col in df.columns:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    return df

def engineer_features(df):
    df = df.copy()
    df['age_bucket']          = pd.cut(df['age'], bins=[0,40,55,65,120], labels=[0,1,2,3]).astype(int)
    df['bp_chol_combined']    = (df['trestbps'] > 130).astype(int) + (df['chol'] > 240).astype(int)
    df['hr_age_ratio']        = (df['thalach'] / df['age']).round(4)
    df['cp_exang_interaction'] = df['cp'] * df['exang']
    return df

df_clean = fill_missing(df)
df_eng   = engineer_features(df_clean)

FEATURES = BASE_FEATURES + ['age_bucket','bp_chol_combined','hr_age_ratio','cp_exang_interaction']

X = df_eng[FEATURES].values
y = df_eng['target'].values

print('Feature matrix shape:', X.shape)
print('Features:', FEATURES)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Before SMOTE:', np.bincount(y_train.astype(int)))
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print('After  SMOTE:', np.bincount(y_train.astype(int)))

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 5 — Train & Evaluate All Classical Models


In [ ]:
MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=2000, C=0.8, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, random_state=42),
    'XGBoost':             xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, verbosity=0, random_state=42),
    'LightGBM':            lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, verbose=-1, random_state=42),
}

results = {}
trained = {}

for name, model in MODELS.items():
    # 5-fold cross-validation
    cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    # Fit and calibrate
    model.fit(X_train, y_train)
    cal = CalibratedClassifierCV(model, method='isotonic', cv=3)
    cal.fit(X_train, y_train)
    trained[name] = cal
    
    # Evaluate
    probs  = cal.predict_proba(X_test)[:, 1]
    preds  = cal.predict(X_test)
    fpr, tpr, _ = roc_curve(y_test, probs)
    
    results[name] = {
        'auc':      roc_auc_score(y_test, probs),
        'accuracy': accuracy_score(y_test, preds),
        'f1':       f1_score(y_test, preds),
        'cv_mean':  scores.mean(),
        'cv_std':   scores.std(),
        'fpr': fpr, 'tpr': tpr
    }
    print(f'{name:25s}  AUC={results[name]["auc"]:.4f}  '
          f'CV={scores.mean():.4f}±{scores.std():.4f}  '
          f'Acc={results[name]["accuracy"]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random (AUC=0.50)')
colors = ['#E53E3E','#3182CE','#38A169','#9B59B6','#D69E2E','#2B6CB0']

for (name, r), c in zip(results.items(), colors):
    ax.plot(r['fpr'], r['tpr'], lw=2, color=c, label=f"{name} (AUC={r['auc']:.3f})")

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

best_name = max(results, key=lambda k: results[k]['auc'])
print(f'\nBest model: {best_name} (AUC = {results[best_name]["auc"]:.4f})')

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
labels = ['No Disease', 'Disease']

for i, (name, model) in enumerate(trained.items()):
    preds = model.predict(X_test)
    cm    = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=labels, yticklabels=labels, cbar=False,
                annot_kws={'size':14,'weight':'bold'})
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6 — PyTorch CardioNet


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

class CardioNet(nn.Module):
    def __init__(self, input_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.LeakyReLU(0.1), nn.Dropout(dropout),
            nn.Linear(64, 32),         nn.LeakyReLU(0.1),
            nn.Linear(32, 1)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

In [ ]:
def to_tensor(X, y=None):
    Xt = torch.FloatTensor(X).to(device)
    if y is not None:
        yt = torch.FloatTensor(np.asarray(y, dtype=np.float32)).unsqueeze(1).to(device)
        return Xt, yt
    return Xt

X_tr_t, y_tr_t = to_tensor(X_train, y_train)
X_te_t, y_te_t = to_tensor(X_test,  y_test)

loader    = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True, drop_last=True)
model_dnn = CardioNet(X_train.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model_dnn.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150, eta_min=1e-6)

history = {'train_loss':[], 'val_loss':[], 'val_auc':[]}
best_auc, best_state, patience = 0, None, 0

for epoch in range(150):
    model_dnn.train()
    e_loss = 0
    for Xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model_dnn(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_dnn.parameters(), 1.0)
        optimizer.step()
        e_loss += loss.item()

    model_dnn.eval()
    with torch.no_grad():
        vl    = criterion(model_dnn(X_te_t), y_te_t).item()
        vprob = torch.sigmoid(model_dnn(X_te_t)).cpu().numpy().flatten()
        vauc  = roc_auc_score(y_test, vprob)

    history['train_loss'].append(e_loss / len(loader))
    history['val_loss'].append(vl)
    history['val_auc'].append(vauc)
    scheduler.step()

    if vauc > best_auc + 0.001:
        best_auc   = vauc
        best_state = {k: v.clone() for k, v in model_dnn.state_dict().items()}
        patience   = 0
    else:
        patience += 1
    if patience >= 20:
        print(f'Early stopping at epoch {epoch}')
        break

model_dnn.load_state_dict(best_state)
print(f'Best Val AUC: {best_auc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train Loss', color='#3182CE')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='#E53E3E', ls='--')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].legend()

axes[1].plot(history['val_auc'], label='Val AUC', color='#38A169')
axes[1].axhline(best_auc, color='gray', ls=':', lw=1, label=f'Best: {best_auc:.4f}')
axes[1].set_title('Validation AUC', fontweight='bold')
axes[1].legend()

plt.suptitle('CardioNet Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Final test evaluation
model_dnn.eval()
with torch.no_grad():
    dnn_probs = torch.sigmoid(model_dnn(X_te_t)).cpu().numpy().flatten()
dnn_preds = (dnn_probs >= 0.5).astype(int)

print('\nCardioNet Final Evaluation:')
print(f'  AUC:      {roc_auc_score(y_test, dnn_probs):.4f}')
print(f'  Accuracy: {accuracy_score(y_test, dnn_preds):.4f}')
print(f'  F1:       {f1_score(y_test, dnn_preds):.4f}')
print(classification_report(y_test, dnn_preds, target_names=['No Disease','Disease']))

## 7 — SHAP Explainability


In [ ]:
# Use Random Forest — TreeExplainer is fastest for tree models
rf_model   = trained['Random Forest']
explainer  = shap.TreeExplainer(rf_model)
shap_vals  = explainer.shap_values(X_test)

# For binary classification, take class 1 values
sv = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_test, feature_names=FEATURES, show=False)
plt.title('SHAP Summary — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Explain one patient
patient_idx = 0
patient_sv  = sv[patient_idx]
pred_prob   = rf_model.predict_proba(X_test[[patient_idx]])[0][1]

top_idx    = np.argsort(np.abs(patient_sv))[-10:][::-1]
top_names  = [FEATURES[i] for i in top_idx]
top_vals   = patient_sv[top_idx]

colors = ['#E53E3E' if v > 0 else '#38A169' for v in top_vals]

plt.figure(figsize=(9, 5))
plt.barh(range(len(top_vals)), top_vals[::-1], color=colors[::-1])
plt.yticks(range(len(top_vals)), top_names[::-1])
plt.axvline(0, color='black', lw=0.8, ls='--')
plt.xlabel('SHAP Value')
plt.title(f'Patient {patient_idx} — Predicted Risk: {pred_prob*100:.1f}%\nTop 10 Contributing Features', fontweight='bold')
plt.tight_layout()
plt.show()

## 8 — Single Patient Prediction Walkthrough


In [ ]:
# Simulate a high-risk patient
test_patient = {
    'age':63, 'sex':1, 'cp':3, 'trestbps':150, 'chol':320,
    'fbs':1, 'restecg':1, 'thalach':120, 'exang':1,
    'oldpeak':3.2, 'slope':2, 'ca':2, 'thal':3
}

row = pd.DataFrame([test_patient])
row['age_bucket']           = pd.cut(row['age'], bins=[0,40,55,65,120], labels=[0,1,2,3]).astype(int)
row['bp_chol_combined']     = (row['trestbps'] > 130).astype(int) + (row['chol'] > 240).astype(int)
row['hr_age_ratio']         = (row['thalach'] / row['age']).round(4)
row['cp_exang_interaction'] = row['cp'] * row['exang']

X_patient = scaler.transform(row[FEATURES].values)

print('=== Predictions from all models ===')
for name, model in trained.items():
    prob = model.predict_proba(X_patient)[0][1]
    risk = 'HIGH' if prob >= 0.66 else 'MEDIUM' if prob >= 0.33 else 'LOW'
    print(f'  {name:25s}: {prob*100:.1f}%  [{risk}]')

## 9 — Final Model Comparison Table


In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'Model':   name,
        'AUC':     round(r['auc'], 4),
        'Accuracy':round(r['accuracy'], 4),
        'F1':      round(r['f1'], 4),
        'CV AUC':  f"{r['cv_mean']:.4f} ± {r['cv_std']:.4f}"
    })

# Add DNN row
model_dnn.eval()
with torch.no_grad():
    dp = torch.sigmoid(model_dnn(X_te_t)).cpu().numpy().flatten()
rows.append({
    'Model':   'CardioNet (PyTorch DNN)',
    'AUC':     round(roc_auc_score(y_test, dp), 4),
    'Accuracy':round(accuracy_score(y_test, (dp>=0.5).astype(int)), 4),
    'F1':      round(f1_score(y_test, (dp>=0.5).astype(int)), 4),
    'CV AUC':  'N/A (single run)'
})

summary = pd.DataFrame(rows).sort_values('AUC', ascending=False).reset_index(drop=True)
print('\nFinal Model Rankings:')
print(summary.to_string(index=False))